# Experimentos de selección de variables

Compara conjuntos de predictores sobre el Parquet EGIF. **No modifica** el datacubo ni los Parquet. El test de 2023 está prohibido en esta fase: se entrena con 2019–2021 y se elige con 2022.

In [1]:
from datetime import datetime, timezone
import json
from pathlib import Path

import pandas as pd

from src.modeling.data import (
    TARGET_COLUMN, TRAIN_YEARS, VALIDATION_YEARS,
    load_dataset_contract, sample_years_for_training,
)
from src.modeling.experiments import fit_lightgbm_experiment
from src.modeling.features import FEATURE_SET_REMOVALS, resolve_feature_set

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
OUTPUT_DIR = ROOT / 'outputs' / 'modeling'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

contract = load_dataset_contract(ROOT / 'data' / 'processed' / 'tabular' / 'egif')
assert tuple(sorted(contract.annual_files)) == ('2019', '2020', '2021', '2022', '2023')
print(f'Predictores publicados: {len(contract.predictors)}')
print('Train:', TRAIN_YEARS, '| Validación:', VALIDATION_YEARS, '| Test bloqueado: (2023,)')

Predictores publicados: 51
Train: (2019, 2020, 2021) | Validación: (2022,) | Test bloqueado: (2023,)


## 1. Muestras reproducibles

Se conservan todas las igniciones y uno de cada 25 negativos, mediante `cell_id % 25`. La prevalencia de estas muestras está enriquecida: sirve para comparar variantes de forma idéntica, no para comunicar la incidencia real.

In [2]:
NEGATIVE_CELL_MODULUS = 25
columns = contract.predictors

train = sample_years_for_training(contract, TRAIN_YEARS, columns, NEGATIVE_CELL_MODULUS)
validation = sample_years_for_training(contract, VALIDATION_YEARS, columns, NEGATIVE_CELL_MODULUS)

for name, frame in {'train': train, 'validation_2022': validation}.items():
    print(f'{name}: {len(frame):,} filas | {int(frame[TARGET_COLUMN].sum()):,} igniciones | ' 
          f'{100 * frame[TARGET_COLUMN].mean():.3f}% positivos')
assert pd.to_datetime(train['fecha']).dt.year.isin(TRAIN_YEARS).all()
assert (pd.to_datetime(validation['fecha']).dt.year == 2022).all()

train: 1,298,221 filas | 4,000 igniciones | 0.308% positivos
validation_2022: 432,668 filas | 1,659 igniciones | 0.383% positivos


## 2. Comparación controlada

Los conjuntos son hipótesis, no decisiones. Se aplica el mismo modelo, semilla y muestreo a todos. No editar esta celda para mirar 2023.

In [3]:
SEED = 42
results = []
models = {}
for feature_set in FEATURE_SET_REMOVALS:
    predictors = resolve_feature_set(contract.predictors, feature_set)
    model, metrics = fit_lightgbm_experiment(train, validation, predictors, seed=SEED)
    models[feature_set] = model
    results.append({'feature_set': feature_set, 'n_predictors': len(predictors), **metrics})

comparison = pd.DataFrame(results).sort_values('pr_auc', ascending=False).reset_index(drop=True)
display(comparison)
assert '2023' not in {str(year) for year in TRAIN_YEARS + VALIDATION_YEARS}

,feature_set,n_predictors,roc_auc,pr_auc,recall_at_top_fraction,top_fraction
0,temporal_compacto,44,0.854522,0.035484,0.043400,0.01
1,completo,51,0.813670,0.024788,0.033153,0.01
2,sin_topografia_redundante,49,0.770153,0.016576,0.052441,0.01


## 3. Registro de resultados

El JSON es la entrada trazable para documentar la decisión. Antes de fijar un conjunto oficial hay que revisar estabilidad, coste operacional e interpretabilidad; el ganador de una sola métrica no se adopta automáticamente.

In [4]:
payload = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'target': TARGET_COLUMN,
    'train_years': list(TRAIN_YEARS),
    'validation_years': list(VALIDATION_YEARS),
    'test_years_not_used': [2023],
    'negative_cell_modulus': NEGATIVE_CELL_MODULUS,
    'seed': SEED,
    'metrics_note': 'Muestra enriquecida: comparar variantes, no interpretar PR-AUC como incidencia poblacional.',
    'results': comparison.to_dict(orient='records'),
}
output_path = OUTPUT_DIR / 'feature_selection_2022.json'
output_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
print(f'Resultados guardados en: {output_path}')

Resultados guardados en: C:\Users\alfon\PycharmProjects\Sistema-deteccion-incendios\outputs\modeling\feature_selection_2022.json
